<h1>RAZ Systems </h1>

Problem Statement: Perplexity-Style Grounded QA System (AutoGen Runtime)
Context

You are building a question-answering system using AutoGen SingleThreadedAgentRuntime(). The system receives a user question and a set of retrieved documents containing factual information with unique IDs and source URLs.

Each document follows this structure:

ID: unique identifier for the fact (e.g., [1], [2], [3])
FACT: the content of the information
SOURCE: the original URL of the information

Using SingleThreadedAgentRuntime() in AutoGen, design a system that ensures:

The model answers the user question only using provided sources
Every claim in the answer is supported by at least one valid source ID
Each answer includes correct inline citations in the form [ID]
A final Sources section is always included, mapping:
[ID] → exact SOURCE URL
The system does not allow hallucinated facts or fabricated citations
If the answer cannot be derived from the sources, the system must explicitly indicate that the information is not supported

A user question
A list of retrieved documents with:
ID
FACT
SOURCE URL

In [ ]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
import requests
import os
from dotenv import load_dotenv

load_dotenv(override=True)


### First we define our Message object

Whatever structure we want for messages in our Agent framework.

In [ ]:
# Let's have a simple one!

@dataclass
class Message:
    content: str


### Now we define our Agent

A subclass of RoutedAgent.

Every Agent has an **Agent ID** which has 2 components:  
`agent.id.type` describes the kind of agent it is  
`agent.id.key` gives it its unique identifier

Any method with the `@message_handler` decorated will have the opportunity to receive messages.


In [ ]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
)
        

Serper Web Search Tool

We’ll wrap Serper into a reusable tool.


### OK let's create a Standalone runtime and register our agent type

In [ ]:

SERPER_API_KEY = os.getenv("SERPER_API_KEY")


def web_search(query: str, k: int = 5):
    url = "https://google.serper.dev/search"

    payload = {"q": query, "num": k}
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    res = requests.post(url, json=payload, headers=headers)
    data = res.json()

    results = []
    for i, item in enumerate(data.get("organic", [])[:k], 1):
        results.append({
            "id": i,
            "title": item.get("title"),
            "link": item.get("link"),
            "snippet": item.get("snippet"),
        })
    
    return results

### Alright! Let's start a runtime and send a message

In [ ]:
web_search("What are AI agents used for in finance?")

In [ ]:
def build_context(results):
    blocks = []

    for r in results:
        blocks.append({
            "id": r["id"],
            "fact": r["snippet"],
            "url": r["link"],
            "title": r["title"]
        })

    return blocks

In [ ]:
def format_for_llm(blocks):
    text = []
    for b in blocks:
        text.append(
            f"ID: [{b['id']}] FACT: {b['fact']}\nSOURCE: {b['url']}\n"
        )
    return "\n".join(text)

In [ ]:
results = web_search("What are AI agents used for in finance?")
    # 2. structure
blocks = build_context(results)

# 3. format for LLM
context = format_for_llm(blocks)

context

Now we integrate it into your SingleThreadedAgentRuntime agent.

In [ ]:

from autogen_core.models import SystemMessage
from autogen_core.models import UserMessage

class ResearchAgent(RoutedAgent):

    def __init__(self, model_client):
        super().__init__("ResearchAgent")
        self.model_client = model_client

    @message_handler
    async def handle(self, message: Message, ctx: MessageContext) -> Message:

        query = message.content

        # 1️⃣ SEARCH
        results = web_search(query)
            # 2. structure
        blocks = build_context(results)

        # 3. format for LLM
        context = format_for_llm(blocks)

        # 2️⃣ LLM PROMPT (IMPORTANT PART)
        prompt = f"""
                You are a STRICT evidence-based research assistant (Perplexity-style).

                ========================
                GOAL
                ========================
                Answer the question ONLY using the provided sources.

                ========================
                SOURCES
                ========================
                Each source has:
                - ID
                - FACT
                - SOURCE

                {context}

                ========================
                QUESTION
                ========================
                {query}

                ========================
                STRICT RULES
                ========================
                1. Use ONLY the provided sources.
                2. Do NOT use external knowledge.
                3. Every bullet MUST include citation like [1], [2].
                4. Do NOT fabricate citations.
                5. If sources do not contain the answer, say:
                "Not supported by provided sources"

                ========================
                OUTPUT FORMAT (STRICT)
                ========================

                A) ANSWER SECTION:
                - Bullet points only
                - Each bullet must end with citation [ID]
                - Each bullet = one idea only

                B) SOURCES SECTION (MANDATORY):
                At the end of the response, always include:

                Sources:
                [1] SOURCE
                [2] SOURCE
                [3] SOURCE

                Rules for Sources section:
                - Must include ALL source IDs used in the answer
                - Must map ID → exact SOURCE URL from context
                - Do NOT omit this section
                - Do NOT modify URLs
                - Do NOT add explanations

                ========================
                FINAL CHECK
                ========================
                Before responding, verify:
                - Every bullet has [ID]
                - Every used ID appears in Sources section
                - Sources section contains correct URLs
                """

        # 3️⃣ CALL GPT-4o-mini
        response = await self.model_client.create([
            SystemMessage(content="You are a strict citation-grounded assistant."),
            UserMessage(content=prompt, source= "user"),
        ])

        answer = response.content

        return Message(content=answer)


Register in Runtime

In [ ]:
runtime = SingleThreadedAgentRuntime()

await ResearchAgent.register(
    runtime,
    "research_agent",
    lambda: ResearchAgent(model_client)
)

runtime.start()

agent_id = AgentId("research_agent", "default")

Test real web search

In [ ]:
response = await runtime.send_message(
    Message("What are AI agents used for in finance?"),
    agent_id
)

print(response.content)

It stops the runtime from processing new work.

In [ ]:
await runtime.stop()
await runtime.close()

### OK Now let's do something more interesting

We'll use an AgentChat Assistant!

In [ ]:

class MyLLMAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("LLMAgent")
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent("LLMAgent", model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received message: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        reply = response.chat_message.content
        print(f"{self.id.type} responded: {reply}")
        return Message(content=reply)
    


In [ ]:


class SimpleAgent(RoutedAgent):

    def __init__(self):
        super().__init__("Simple Agent")

    @message_handler
    async def handle_message(self, message: Message, ctx: MessageContext) -> Message:
        print(f"Simple Agent Received: {message.content}")

        return Message(
            content=f"simple Agent Response: {message.content}"
        )

In [ ]:
from autogen_core import SingleThreadedAgentRuntime

runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())
await MyLLMAgent.register(runtime, "LLMAgent", lambda: MyLLMAgent())

In [ ]:
runtime.start()  # Start processing messages in the background.
response = await runtime.send_message(Message("Hi there!"), AgentId("LLMAgent", "default"))
print(">>>", response.content)
response =  await runtime.send_message(Message(response.content), AgentId("simple_agent", "default"))
print(">>>", response.content)
response = await runtime.send_message(Message(response.content), AgentId("LLMAgent", "default"))

In [ ]:
await runtime.stop()
await runtime.close()

### OK now let's show this at work - let's have 3 agents interact!

In [ ]:
from autogen_ext.models.ollama import OllamaChatCompletionClient


class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)

In [ ]:
JUDGE = "You are judging a game of rock, paper, scissors. The players have made these choices:\n"

class RockPaperScissorsAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        instruction = "You are playing rock, paper, scissors. Respond only with the one word, one of the following: rock, paper, or scissors."
        message = Message(content=instruction)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message, inner_1)
        response2 = await self.send_message(message, inner_2)
        result = f"Player 1: {response1.content}\nPlayer 2: {response2.content}\n"
        judgement = f"{JUDGE}{result}Who wins?"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)


In [ ]:
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await RockPaperScissorsAgent.register(runtime, "rock_paper_scissors", lambda: RockPaperScissorsAgent("rock_paper_scissors"))
runtime.start()

In [ ]:
agent_id = AgentId("rock_paper_scissors", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(response.content)

In [ ]:
await runtime.stop()
await runtime.close()